# VLM আর্কিটেকচার ও Fusion কৌশল

VLM fusion strategies — একটি language model-এ vision ঢোকানোর তিনটি উপায়, সবগুলোই scratch থেকে implement করা এবং একই toy grounding কাজে train করা।

কোনো download নেই, কোনো pretrained weight নেই। CPU-তে ~1 মিনিট।

**কাজটি** (ইচ্ছাকৃতভাবে ছোট কিন্তু সত্যিই non-trivial একটি grounding সমস্যা):

একটি "scene" হলো `NUM_OBJECTS` টি বস্তু। প্রতিটি বস্তুর একটি shape ও একটি color আছে, এবং প্রতিটি model-কে দেওয়া হয় একটি vision token হিসেবে (দুটোকেই encode করা একটি noisy vector)। প্রশ্নটি একটি shape-এর নাম বলে — `"what color is the <shape>?"` — এবং model-কে সেই বস্তুর color output করতে হবে। যে model image উপেক্ষা করে সে chance-এ আটকে থাকে, আর যে model image-কে একটি vector-এ pool করে সে কেবল অনুমানটিকে "scene-এর কোথাও উপস্থিত color-গুলো"-তে সংকুচিত করতে পারে: কাজটি সমাধান করতে হলে queried shape ধারণকারী **একটি** token-এ attend করতে হবে এবং তার সাথে bound color-টি পড়ে নিতে হবে। এটিই grounding, ক্ষুদ্রাকারে।

**তুলনা করা তিনটি architecture:**

- **A. PREFIX / PROJECTOR fusion** (LLaVA, বেশিরভাগ open VLM) — vision token-গুলো LLM-এর embedding space-এ project করা হয় এবং text token-গুলোর সামনে **CONCATENATE** করা হয়; বাকিটা self-attention করে।
- **B. CROSS-ATTENTION fusion** (Flamingo, Llama-3-V, Idefics-style) — text stream নিজের দৈর্ঘ্য বজায় রাখে; অতিরিক্ত gated cross-attention layer-গুলো text token-কে vision token থেকে পড়তে দেয়।
- **C. EARLY / NATIVE fusion** (Chameleon, Fuyu-style) — আলাদা কোনো pretrained tower interface নেই; raw patch vector-গুলো প্রথম ধাপ থেকেই token হিসেবে সরাসরি shared Transformer-এ যায়।

প্রতিটির জন্য মাপা হয়: held-out accuracy, parameter count, text sequence length, attention cost, এবং — frozen-LLM setting-এ — language model-এর কতটা অংশ আদৌ ছুঁতে হয়।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান (CPU-তে মোট ~1 মিনিট)। Section 0–2 কেবল data ও model সংজ্ঞায়িত করে; প্রকৃত training ও ফলাফল আসে Section 3 থেকে। Section 5 (zero-init gate) Section 3-এ train করা cross-attention model ব্যবহার করে, তাই আগের cell-গুলো চালানো থাকতে হবে।

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

NUM_SHAPES = 8
NUM_COLORS = 6
NUM_OBJECTS = 6           # প্রতি scene-এ vision token-সংখ্যা
D_VISION = 32             # "vision tower"-এর output width
D_MODEL = 64              # LLM-এর hidden width
N_TEXT_TOKENS = 4         # প্রশ্নটি, text token হিসেবে

## 0. Synthetic scene ও প্রশ্ন

প্রতিটি shape ও color-এর জন্য একটি স্থির random "appearance" code তৈরি করা হয়। একটি বস্তুর vision token হলো তার shape code ও color code-এর **যোগফল** (সাথে কিছু noise), তাই একটি token-এ দুটো তথ্যই থাকে এবং model-কে সেগুলো আলাদা করতে হয়। `make_batch` প্রতিটি scene-এ ভিন্ন ভিন্ন shape রাখে যাতে প্রশ্নটি দ্ব্যর্থহীন থাকে, এবং প্রশ্নকে চারটি token id হিসেবে encode করে: `[WHAT] [COLOR] [IS] [<shape>]`।

In [ ]:
# ---------------------------------------------------------------------------
# 0. Synthetic scene ও প্রশ্ন
# ---------------------------------------------------------------------------

# স্থির random "appearance" code: raw vision-feature space-এ একটি shape ও একটি
# color দেখতে কেমন। দুটো যোগ (ADD) করা হয়, তাই একটি token-ই দুটো বহন করে এবং
# model-কে সেগুলো আলাদা (disentangle) করতে হয়।
shape_code = torch.randn(NUM_SHAPES, D_VISION)
color_code = torch.randn(NUM_COLORS, D_VISION)


def make_batch(batch_size):
    """Vision token, প্রশ্নের token id, এবং উত্তরের color id ফেরত দেয়।"""
    # প্রতিটি scene-এ NUM_OBJECTS টি ভিন্ন shape থাকে, তাই প্রশ্নটি দ্ব্যর্থহীন।
    shapes = torch.stack([torch.randperm(NUM_SHAPES)[:NUM_OBJECTS] for _ in range(batch_size)])
    colors = torch.randint(0, NUM_COLORS, (batch_size, NUM_OBJECTS))
    vision = shape_code[shapes] + color_code[colors]                    # (B, N_obj, D_VISION)
    vision = vision + 0.25 * torch.randn_like(vision)

    # প্রশ্ন: উপস্থিত বস্তুগুলোর মধ্যে কোনটির কথা জিজ্ঞাসা করা হচ্ছে?
    target_slot = torch.randint(0, NUM_OBJECTS, (batch_size,))
    queried_shape = shapes[torch.arange(batch_size), target_slot]       # (B,)
    answer = colors[torch.arange(batch_size), target_slot]              # (B,)

    # প্রশ্নের token id: [WHAT] [COLOR] [IS] [<shape>]  -- shape token id-টি
    # 3টি স্থির vocabulary শব্দের পরে offset করা।
    fixed = torch.tensor([0, 1, 2]).expand(batch_size, 3)
    question = torch.cat([fixed, (queried_shape + 3).unsqueeze(1)], dim=1)   # (B, 4)
    return vision, question, answer


TEXT_VOCAB = 3 + NUM_SHAPES

## 1. Shared building block (একটি minimal Transformer, Phase 02-এর মতো)

তিনটি architecture-ই একই উপাদান দিয়ে তৈরি:

- `SelfAttentionBlock` — একটি সাধারণ pre-LN Transformer block।
- `GatedCrossAttentionBlock` — Flamingo-র সংযোজন: text থেকে vision-এ cross-attention, একটি **zero-initialized `tanh` gate** দিয়ে মোড়ানো, যাতে initialization-এ block-টি কিছুই না বদলায়।
- `TextBackbone` — "pretrained LLM"-এর stand-in; তিনটি model-এই একই body।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Shared building block (একটি minimal Transformer, Phase 02-এর মতো)
# ---------------------------------------------------------------------------

class SelfAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.ff(self.ln2(x))


class GatedCrossAttentionBlock(nn.Module):
    """Flamingo-র সংযোজন: text থেকে vision-এ cross-attention, শূন্যে
    initialize করা একটি TANH GATE-এ মোড়ানো, যাতে initialization-এ block-টি
    একটি no-op হয়।

    এই খুঁটিনাটিটিই মূল কারণ যে একটি frozen pretrained LLM এই layer-গুলো
    ঢোকানোর পরও টিকে থাকে: step 0-এ পরিবর্তিত model-টি bit-for-bit ঠিক তা-ই
    গণনা করে যা মূল model করত, এবং training যতটা সার্থক করে gate কেবল
    ততটাই খোলে।
    """

    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.ln_q = nn.LayerNorm(d_model)
        self.ln_kv = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.gate = nn.Parameter(torch.zeros(1))          # <- zero-init gate
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                nn.Linear(4 * d_model, d_model))
        self.ff_gate = nn.Parameter(torch.zeros(1))
        self.ln_ff = nn.LayerNorm(d_model)

    def forward(self, text, vision):
        a = self.attn(self.ln_q(text), self.ln_kv(vision), self.ln_kv(vision),
                      need_weights=False)[0]
        text = text + torch.tanh(self.gate) * a
        return text + torch.tanh(self.ff_gate) * self.ff(self.ln_ff(text))


class TextBackbone(nn.Module):
    """'Pretrained LLM'-এর stand-in। তিনটি model-এই একই body।"""

    def __init__(self, n_layers=3):
        super().__init__()
        self.tok_embed = nn.Embedding(TEXT_VOCAB, D_MODEL)
        self.pos_embed = nn.Parameter(torch.randn(1, 64, D_MODEL) * 0.02)
        self.blocks = nn.ModuleList([SelfAttentionBlock(D_MODEL) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, NUM_COLORS)

## 2. তিনটি architecture (ও দুটি control)

এখানে পাঁচটি model সংজ্ঞায়িত হয়:

- `PrefixVLM` (A) — vision token project করে text-এর সামনে জুড়ে দেয়; sequence length = 6 + 4।
- `CrossAttnVLM` (B) — text stream-এর দৈর্ঘ্য 4-ই থাকে; প্রতিটি layer-এর আগে একটি gated cross-attention block image পড়ে।
- `EarlyFusionVLM` (C) — একটি single linear "tokenizer" ও modality embedding, একটি Transformer।
- `BlindBaseline` — কখনো image দেখে না; chance-level control।
- `PooledVLM` — Lesson 1-এর pooling যুক্তির control: image-কে LLM দেখার আগেই **একটি** vector-এ চেপে ফেলা হয়, ঠিক একটি CLIP embedding-এর মতো।

প্রতিটি model-এর `seq_len()` LLM-এর self-attention যে sequence দেখে তার দৈর্ঘ্য ফেরত দেয়, যা পরে attention cost গণনায় লাগে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. তিনটি architecture
# ---------------------------------------------------------------------------

class PrefixVLM(nn.Module):
    """A. LLaVA-style: vision token-গুলোকে LLM-এর space-এ project করে সামনে জুড়ে দাও।"""

    def __init__(self):
        super().__init__()
        self.lm = TextBackbone()
        self.projector = nn.Sequential(nn.Linear(D_VISION, D_MODEL), nn.GELU(),
                                       nn.Linear(D_MODEL, D_MODEL))

    def forward(self, vision, question):
        vis = self.projector(vision)                                   # (B, N_obj, D_MODEL)
        txt = self.lm.tok_embed(question)                              # (B, N_text, D_MODEL)
        x = torch.cat([vis, txt], dim=1)                               # একটিই sequence
        x = x + self.lm.pos_embed[:, : x.shape[1]]
        for blk in self.lm.blocks:
            x = blk(x)
        return self.lm.head(self.lm.ln_f(x[:, -1]))                    # শেষ token-টি পড়ো

    def seq_len(self):
        return NUM_OBJECTS + N_TEXT_TOKENS


class CrossAttnVLM(nn.Module):
    """B. Flamingo-style: text stream নিজের দৈর্ঘ্য রাখে; gated cross-attention
    layer-গুলো মাঝে মাঝে বসানো থাকে যাতে text token-গুলো vision token থেকে পড়তে পারে।"""

    def __init__(self):
        super().__init__()
        self.lm = TextBackbone()
        self.vision_proj = nn.Linear(D_VISION, D_MODEL)
        self.xattn = nn.ModuleList([GatedCrossAttentionBlock(D_MODEL)
                                    for _ in range(len(self.lm.blocks))])

    def forward(self, vision, question):
        vis = self.vision_proj(vision)
        x = self.lm.tok_embed(question) + self.lm.pos_embed[:, :question.shape[1]]
        for blk, xa in zip(self.lm.blocks, self.xattn):
            x = xa(x, vis)                                             # image পড়ো
            x = blk(x)                                                 # তারপর text-এ চিন্তা করো
        return self.lm.head(self.lm.ln_f(x[:, -1]))

    def seq_len(self):
        return N_TEXT_TOKENS


class EarlyFusionVLM(nn.Module):
    """C. Native/early fusion: একটি Transformer, raw patch vector-ই token,
    আলাদাভাবে pretrained কোনো tower interface নেই এবং কিছুই frozen নয়।"""

    def __init__(self):
        super().__init__()
        self.lm = TextBackbone()
        self.patch_in = nn.Linear(D_VISION, D_MODEL)     # একটি single linear "tokenizer"
        self.modality_embed = nn.Parameter(torch.randn(2, D_MODEL) * 0.02)

    def forward(self, vision, question):
        vis = self.patch_in(vision) + self.modality_embed[0]
        txt = self.lm.tok_embed(question) + self.modality_embed[1]
        x = torch.cat([vis, txt], dim=1)
        x = x + self.lm.pos_embed[:, : x.shape[1]]
        for blk in self.lm.blocks:
            x = blk(x)
        return self.lm.head(self.lm.ln_f(x[:, -1]))

    def seq_len(self):
        return NUM_OBJECTS + N_TEXT_TOKENS


class BlindBaseline(nn.Module):
    """শুধু text -- কখনো image দেখে না। Chance-level control।"""

    def __init__(self):
        super().__init__()
        self.lm = TextBackbone()

    def forward(self, vision, question):
        x = self.lm.tok_embed(question) + self.lm.pos_embed[:, :question.shape[1]]
        for blk in self.lm.blocks:
            x = blk(x)
        return self.lm.head(self.lm.ln_f(x[:, -1]))

    def seq_len(self):
        return N_TEXT_TOKENS


class PooledVLM(nn.Module):
    """Lesson 1-এর pooling যুক্তির একটি control: LLM দেখার আগেই image-কে
    ONE vector-এ চেপে ফেলা হয়, ঠিক একটি CLIP embedding-এর মতো।"""

    def __init__(self):
        super().__init__()
        self.lm = TextBackbone()
        self.projector = nn.Sequential(nn.Linear(D_VISION, D_MODEL), nn.GELU(),
                                       nn.Linear(D_MODEL, D_MODEL))

    def forward(self, vision, question):
        vis = self.projector(vision).mean(dim=1, keepdim=True)         # (B, 1, D_MODEL)
        txt = self.lm.tok_embed(question)
        x = torch.cat([vis, txt], dim=1)
        x = x + self.lm.pos_embed[:, : x.shape[1]]
        for blk in self.lm.blocks:
            x = blk(x)
        return self.lm.head(self.lm.ln_f(x[:, -1]))

    def seq_len(self):
        return 1 + N_TEXT_TOKENS

## 3. Training ও evaluation: সব layer trainable

`train_and_eval` একটি model-কে 700 step train করে, তারপর 2000টি held-out scene-এ accuracy মাপে এবং মোট ও trainable parameter count ফেরত দেয়। `freeze_lm=True` দিলে language model-এর weight freeze করা হয় (কেবল output head বাদে) — Section 4 এটি ব্যবহার করে।

এই cell পাঁচটি model-ই সব layer trainable রেখে train করে এবং accuracy, parameter, text sequence length ও attention cell (`L * L`) দেখায়। Train করা model-গুলো `trained` dict-এ রাখা হয়, যা Section 5 আবার ব্যবহার করে।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Training ও evaluation
# ---------------------------------------------------------------------------

def train_and_eval(model_cls, steps=700, batch_size=64, lr=2e-3, freeze_lm=False):
    torch.manual_seed(1)
    model = model_cls()
    if freeze_lm:
        for p in model.lm.parameters():
            p.requires_grad = False
        # Head-কে তবুও color emit করা শিখতে হয়; একটি বাস্তব VLM-এ LLM-এর
        # বিদ্যমান output head আগে থেকেই color শব্দগুলো জানে।
        for p in model.lm.head.parameters():
            p.requires_grad = True
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(trainable, lr=lr)
    for _ in range(steps):
        vision, question, answer = make_batch(batch_size)
        loss = F.cross_entropy(model(vision, question), answer)
        opt.zero_grad()
        loss.backward()
        opt.step()

    with torch.no_grad():
        vision, question, answer = make_batch(2000)
        acc = (model(vision, question).argmax(1) == answer).float().mean().item()
    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in trainable)
    return acc, n_total, n_train, model


print("=" * 78)
print("THE TASK: 6 objects, each one vision token carrying shape+color.")
print("Question: 'what color is the <shape>?'  Chance accuracy = 1/6 = 16.7%.")
print("=" * 78)
print()

print("=" * 78)
print("1. ALL LAYERS TRAINABLE")
print("=" * 78)
print(f"{'architecture':<34} {'accuracy':>9} {'params':>10} {'text seq len':>13} {'attn cells':>11}")

configs = [
    ("blind (text only, no image)", BlindBaseline),
    ("pooled image -> 1 token", PooledVLM),
    ("A. prefix / projector (LLaVA)", PrefixVLM),
    ("B. cross-attention (Flamingo)", CrossAttnVLM),
    ("C. early / native fusion", EarlyFusionVLM),
]

trained = {}
for name, cls in configs:
    acc, n_total, n_train, model = train_and_eval(cls)
    trained[name] = (acc, model)
    L = model.seq_len()
    print(f"{name:<34} {acc:>8.1%} {n_total:>10,} {L:>13} {L * L:>11}")

print()
print("The blind model is pinned at chance -- with no image there is nothing to")
print("read the colour off. The pooled model lands well above chance but far")
print("below solving the task: a single averaged vector still carries WHICH")
print("colours are in the scene, so guessing among them beats guessing among")
print("all six, but it has lost which colour is BOUND to which shape. That")
print("binding is what per-token vision features preserve, and it is Lesson 1's")
print("pooling argument restated as task accuracy. All three real fusion")
print("strategies solve the task outright.")
print()
print("Note the cost asymmetry in the last two columns: prefix fusion makes the")
print("LLM's sequence longer (here 6 image tokens + 4 text tokens), and self-")
print("attention is quadratic in that length. Cross-attention leaves the text")
print("stream at its original length and pays for the image separately, in")
print("cross-attention layers whose cost is (text_len x vision_len), i.e. LINEAR")
print("in the number of image tokens. With 576 or 3,136 real vision tokens that")
print("difference is the whole ballgame -- see Lesson 10.")
print()

## 4. Frozen-LLM setting: প্রতিটি কৌশলের খরচ কত

একটি বাস্তব VLM শুরু হয় এমন একটি LLM থেকে যা আগে থেকেই কাজ করে। এটিকে freeze করলে তার text ক্ষমতা সুরক্ষিত থাকে, আর তখন সিদ্ধান্তের প্রশ্ন হয়ে দাঁড়ায়: "এই fusion কৌশলের জন্য কতটা **নতুন** যন্ত্রাংশ লাগে?" এই cell তিনটি fusion architecture-কে `freeze_lm=True` দিয়ে আবার train করে এবং accuracy, trainable parameter ও model-এর কত শতাংশ train হলো তা দেখায়।

In [ ]:
# ---------------------------------------------------------------------------
# 4. Frozen-LLM setting: প্রতিটি কৌশল আপনার কাছে কী খরচ করায়
# ---------------------------------------------------------------------------

print("=" * 78)
print("2. WITH THE LANGUAGE MODEL FROZEN (the realistic starting point)")
print("=" * 78)
print("A real VLM starts from an LLM that already works. Freezing it protects")
print("its text ability, and makes 'how much NEW machinery does this fusion")
print("strategy need?' the deciding question.")
print()
print(f"{'architecture':<34} {'accuracy':>9} {'trainable':>11} {'% of model':>11}")
for name, cls in [("A. prefix / projector (LLaVA)", PrefixVLM),
                  ("B. cross-attention (Flamingo)", CrossAttnVLM),
                  ("C. early / native fusion", EarlyFusionVLM)]:
    acc, n_total, n_train, _ = train_and_eval(cls, freeze_lm=True)
    print(f"{name:<34} {acc:>8.1%} {n_train:>11,} {n_train / n_total:>10.1%}")

print()
print("Prefix fusion trains a tiny projector and nothing else -- that is why it")
print("is the cheapest way to build a VLM and why LLaVA-style recipes dominate")
print("open-source work. Cross-attention adds real parameters (a whole gated")
print("block per layer, ~50% of the model here) but touches no existing weight,")
print("which is why Flamingo could bolt vision onto a frozen 70B LM.")
print()
print("Early fusion is in the table only for completeness: this toy task is easy")
print("enough that a frozen backbone still solves it, but the frozen column does")
print("not really apply to it. Its premise is that the text and image weights")
print("were never separate in the first place -- there is no pretrained language")
print("model sitting there to freeze, and buying its higher ceiling means paying")
print("for a full multimodal pretraining run (Lesson 5).")
print()

## 5. Zero-init gate: কেন cross-attention ঢোকানোর পরও একটি frozen LLM টিকে থাকে

এখানে একটি নতুন `CrossAttnVLM` ও একটি `BlindBaseline` তৈরি করা হয় এবং দুটিতে হুবহু একই language model weight load করা হয়। Initialization-এ দুটির output-এর সর্বোচ্চ পার্থক্য ঠিক শূন্য, কারণ `tanh(0) = 0` — প্রতিটি ঢোকানো block তখন একটি identity। তারপর Section 3-এ train করা cross-attention model-এর gate-এর মান layer অনুযায়ী দেখানো হয়: অ-শূন্য মান মানে model image-কে ঢুকতে দিয়েছে, আর প্রতি layer-এ কতটা ঢুকতে দিয়েছে তা সরাসরি পড়া যায়।

In [ ]:
# ---------------------------------------------------------------------------
# 5. Zero-init gate: কেন cross-attention ঢোকানোর পরও একটি frozen LLM টিকে থাকে
# ---------------------------------------------------------------------------

print("=" * 78)
print("3. THE ZERO-INITIALIZED GATE (Flamingo's key trick)")
print("=" * 78)

torch.manual_seed(1)
xvlm = CrossAttnVLM()
blind = BlindBaseline()
blind.lm.load_state_dict(xvlm.lm.state_dict())      # হুবহু একই language model

vision, question, _ = make_batch(8)
with torch.no_grad():
    out_with_xattn = xvlm(vision, question)
    out_text_only = blind(vision, question)
print(f"max |cross-attn model - text-only model| at init : "
      f"{(out_with_xattn - out_text_only).abs().max().item():.2e}")
print("Exactly zero: tanh(0) = 0, so every inserted block is an identity at")
print("initialization and the frozen LLM's behaviour is preserved bit-for-bit.")
print("Training then opens the gates only as far as the vision signal earns.")

_, gated_model = trained["B. cross-attention (Flamingo)"]
gates = [torch.tanh(b.gate).item() for b in gated_model.xattn]
print(f"gate values after training, layer by layer      : "
      f"{', '.join(f'{g:+.3f}' for g in gates)}")
print("Non-zero after training = the model chose to let the image in, and how")
print("much it let in per layer is a directly readable number.")